# 06 - Dashboard Interactivo con Plotly: El taxi de NYC

## Fase 2 del proyecto NYC Taxi

En este notebook construimos **visualizaciones interactivas** usando Plotly, una librería que genera gráficos con capacidad de zoom, hover, selección y animación directamente en el navegador.

### Objetivos de aprendizaje
- Crear gráficos interactivos con **Plotly Express** (API de alto nivel)
- Construir gráficos personalizados con **plotly.graph_objects** (API de bajo nivel)
- Usar **Scatter Mapbox** para mapas interactivos basados en zonas (sin token de API)
- Combinar múltiples gráficos en un **dashboard con subplots**
- Crear visualizaciones jerárquicas con **Sunburst**

### Diferencia entre Plotly Express y Graph Objects
- **Plotly Express (px)**: API concisa para gráficos rápidos, similar a seaborn
- **Graph Objects (go)**: API completa para personalización total, similar a matplotlib

---
## Carga de datos

In [ ]:
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
print("Plotly y dependencias cargadas correctamente.")

In [ ]:
# Definición de zonas TLC por location_id
MANHATTAN_ZONES = (
    "'4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90',"
    "'100','107','113','114','116','125','127','128','137','140','141','142','143','144',"
    "'148','151','152','153','158','161','162','163','164','166','170','186','194','202',"
    "'209','211','224','229','230','231','232','233','234','236','237','238','239','243',"
    "'244','246','249','261','262','263'"
)

BROOKLYN_ZONES = (
    "'11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52',"
    "'54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91',"
    "'97','106','108','111','112','123','133','149','150','154','155','165','177','178',"
    "'181','188','189','190','195','210','217','222','225','227','228','255','256','257'"
)

query = f"""
SELECT
    pickup_datetime,
    dropoff_datetime,
    pickup_location_id,
    dropoff_location_id,
    passenger_count,
    trip_distance,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount,
    payment_type,
    EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS day_of_week,
    EXTRACT(DATE FROM pickup_datetime) AS pickup_date,
    EXTRACT(MONTH FROM pickup_datetime) AS pickup_month,
    TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) AS trip_duration_min,
    CASE
        WHEN pickup_location_id = '132' THEN 'JFK'
        WHEN pickup_location_id = '138' THEN 'LaGuardia'
        WHEN pickup_location_id = '1'   THEN 'Newark'
        WHEN pickup_location_id IN ({MANHATTAN_ZONES}) THEN 'Manhattan'
        WHEN pickup_location_id IN ({BROOKLYN_ZONES}) THEN 'Brooklyn'
        ELSE 'Queens/Otro'
    END AS pickup_zone,
    SAFE_DIVIDE(tip_amount, fare_amount) * 100 AS tip_percentage
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    fare_amount BETWEEN 2.5 AND 200
    AND trip_distance BETWEEN 0.1 AND 50
    AND passenger_count BETWEEN 1 AND 6
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) BETWEEN 1 AND 180
ORDER BY RAND()
LIMIT 30000
"""

df = bq.query(query)
print(f"Registros cargados: {len(df):,}")

# Mapear día de la semana a nombre
day_names = {1: 'Domingo', 2: 'Lunes', 3: 'Martes', 4: 'Miércoles',
             5: 'Jueves', 6: 'Viernes', 7: 'Sábado'}
df['day_name'] = df['day_of_week'].map(day_names)

# Categoría de propina
df['tip_category'] = pd.cut(
    df['tip_percentage'],
    bins=[-1, 0, 10, 18, 25, 100],
    labels=['Sin propina', 'Baja (0-10%)', 'Estándar (10-18%)',
            'Generosa (18-25%)', 'Muy generosa (>25%)']
)

# Convertir pickup_date a datetime si es string
df['pickup_date'] = pd.to_datetime(df['pickup_date'])

df.head()

---
## 1. Histograma animado de tarifas por hora

Plotly Express permite crear **animaciones** fácilmente con el parámetro `animation_frame`. Aquí visualizamos cómo cambia la distribución de tarifas a lo largo del día.

### Nota
Usa el botón **Play** debajo del gráfico para ver la animación, o arrastra el slider manualmente.

In [ ]:
# Histograma animado de tarifas por hora del día
df_hist = df[df['fare_amount'] <= 60].copy()

fig = px.histogram(
    df_hist,
    x='fare_amount',
    animation_frame='pickup_hour',
    nbins=40,
    range_x=[0, 60],
    range_y=[0, 800],
    color_discrete_sequence=['#2196F3'],
    labels={'fare_amount': 'Tarifa (USD)', 'pickup_hour': 'Hora'},
    title='Distribución de tarifas por hora del día (usa el slider)'
)

fig.update_layout(
    xaxis_title='Tarifa (USD)',
    yaxis_title='Frecuencia',
    template='plotly_white',
    height=500
)

fig.show()

**Observaciones esperadas:**
- Durante las horas pico (8-9 AM, 5-7 PM) hay más viajes en general
- La forma de la distribución se mantiene similar, pero el volumen cambia
- En la madrugada (2-5 AM) hay pocos viajes pero la distribución puede cambiar

---
## 2. Heatmap interactivo: hora x día de la semana

Usamos `plotly.graph_objects.Heatmap` para crear un heatmap interactivo que muestra el conteo de viajes por hora y día de la semana. Al pasar el cursor, se ve el valor exacto.

In [ ]:
# Pivot table: hora x día de la semana
day_order = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']

heatmap_data = df.pivot_table(
    values='fare_amount',
    index='day_name',
    columns='pickup_hour',
    aggfunc='count',
    fill_value=0
)

# Reordenar días
heatmap_data = heatmap_data.reindex(day_order)

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=[f"{h}:00" for h in heatmap_data.columns],
    y=heatmap_data.index,
    colorscale='YlOrRd',
    colorbar=dict(title='Viajes'),
    hovertemplate='Día: %{y}<br>Hora: %{x}<br>Viajes: %{z}<extra></extra>'
))

fig.update_layout(
    title='Mapa de calor: Viajes por hora y día de la semana',
    xaxis_title='Hora del día',
    yaxis_title='Día de la semana',
    template='plotly_white',
    height=400,
    yaxis=dict(autorange='reversed')  # Lunes arriba
)

fig.show()

**Patrones esperados:**
- **Días laborales**: Picos a las 8-9 AM (ida al trabajo) y 5-7 PM (vuelta)
- **Viernes/Sábado noche**: Actividad elevada hasta altas horas
- **Domingo mañana**: El período más tranquilo de la semana
- La escala de color ayuda a identificar rápidamente los períodos de máxima demanda

---
## 3. Scatter Mapbox: zonas de recogida por volumen y tarifa

Plotly permite crear mapas interactivos con `scatter_mapbox` usando tiles de OpenStreetMap, sin necesidad de token de Mapbox.

### Enfoque basado en zonas
En lugar de graficar puntos individuales (que requerirían coordenadas GPS), agregamos los datos por `pickup_location_id` y usamos los centroides de cada Taxi Zone. El tamaño del marcador refleja el volumen de viajes y el color la tarifa promedio.

In [ ]:
# Lookup de centroides aproximados para las zonas TLC
ZONE_CENTROIDS = {
    # Manhattan - Midtown
    '161': (40.7574, -73.9722, 'Midtown Center'),
    '162': (40.7527, -73.9772, 'Midtown East'),
    '163': (40.7620, -73.9847, 'Midtown North'),
    '164': (40.7548, -73.9916, 'Midtown South'),
    '170': (40.7527, -73.9930, 'Murray Hill'),
    '186': (40.7590, -73.9845, 'Penn Station/Madison Sq West'),
    '234': (40.7614, -73.9776, 'Union Sq'),
    '236': (40.7736, -73.9566, 'Upper East Side N'),
    '237': (40.7736, -73.9566, 'Upper East Side S'),
    '238': (40.7870, -73.9537, 'Upper East Side'),
    '239': (40.7957, -73.9712, 'Upper West Side N'),
    '263': (40.7831, -73.9712, 'Upper West Side S'),
    '262': (40.7757, -73.9817, 'Yorkville West'),
    '233': (40.7580, -73.9855, 'Times Square/Theatre'),
    '230': (40.7565, -73.9870, 'Times Square'),
    '48':  (40.7590, -73.9920, 'Clinton East'),
    '50':  (40.7661, -73.9860, 'Clinton West'),
    '142': (40.7728, -73.9870, 'Lincoln Square East'),
    '141': (40.7728, -73.9850, 'Lincoln Square West'),
    '140': (40.7690, -73.9580, 'Lenox Hill East'),
    '137': (40.7690, -73.9630, 'Lenox Hill West'),
    '107': (40.7450, -73.9940, 'Gramercy'),
    '113': (40.7443, -74.0067, 'Greenwich Village N'),
    '114': (40.7340, -74.0030, 'Greenwich Village S'),
    '125': (40.7209, -74.0065, 'Hudson Sq'),
    '158': (40.7220, -73.9880, 'Meatpacking/West Village'),
    '249': (40.7209, -73.9995, 'West Village'),
    '4':   (40.7020, -74.0120, 'Battery Park'),
    '12':  (40.7070, -74.0120, 'Battery Park City'),
    '13':  (40.7100, -73.9950, 'Chinatown'),
    '87':  (40.7131, -74.0097, 'Financial District N'),
    '88':  (40.7060, -74.0131, 'Financial District S'),
    '209': (40.7210, -73.9990, 'SoHo'),
    '231': (40.7190, -73.9888, 'TriBeCa/Civic Center'),
    '144': (40.7230, -73.9870, 'Lower East Side'),
    '232': (40.7290, -73.9870, 'East Village'),
    '79':  (40.7940, -73.9400, 'East Harlem N'),
    '116': (40.7840, -73.9440, 'East Harlem S'),
    '128': (40.7460, -73.9780, 'Kips Bay'),
    '127': (40.7510, -73.9690, 'Turtle Bay/UN'),
    '100': (40.7535, -73.9880, 'Garment District'),
    '68':  (40.7980, -73.9420, 'East Harlem'),
    '90':  (40.7417, -73.9893, 'Flatiron'),
    '246': (40.7510, -74.0020, 'West Chelsea/Hudson Yards'),
    '243': (40.8130, -73.9430, 'Central Harlem N'),
    '244': (40.8020, -73.9480, 'Central Harlem S'),
    '152': (40.7440, -73.9930, 'Midtown (other)'),
    '166': (40.8034, -73.9560, 'Morningside Heights'),
    '45':  (40.7460, -73.9990, 'Chelsea'),
    '74':  (40.7420, -73.9750, 'East Chelsea'),
    '75':  (40.7560, -73.9680, 'East Midtown'),
    '24':  (40.7240, -73.9800, 'Alphabet City'),
    '41':  (40.7710, -73.9740, 'Central Park'),
    '42':  (40.7851, -73.9683, 'Central Park W'),
    '43':  (40.7740, -73.9676, 'Central Park E'),
    '148': (40.7670, -73.9500, 'Sutton Place'),
    '151': (40.7480, -73.9780, 'Murray Hill S'),
    '153': (40.7380, -73.9850, 'Stuyvesant Town'),
    '194': (40.8090, -73.9530, 'Harlem'),
    '202': (40.8160, -73.9580, 'Hamilton Heights'),
    '211': (40.8090, -73.9630, 'Sugar Hill'),
    '224': (40.8010, -73.9680, 'Manhattanville'),
    '229': (40.7250, -74.0080, 'Tribeca S'),
    '261': (40.8220, -73.9500, 'Washington Heights N'),
    # Brooklyn
    '61':  (40.6880, -73.9795, 'Crown Heights N'),
    '25':  (40.6850, -73.9830, 'Boerum Hill'),
    '33':  (40.6950, -73.9950, 'Brooklyn Heights'),
    '67':  (40.6990, -73.9870, 'DUMBO/Downtown Bklyn'),
    '76':  (40.6900, -73.9750, 'Fort Greene'),
    '85':  (40.7270, -73.9520, 'Greenpoint'),
    '97':  (40.6720, -73.9780, 'Park Slope'),
    '150': (40.7111, -73.9545, 'Williamsburg N'),
    '256': (40.7077, -73.9568, 'Williamsburg S'),
    '112': (40.6760, -74.0050, 'Red Hook'),
    '52':  (40.6890, -73.9660, 'Clinton Hill'),
    '106': (40.6770, -73.9680, 'Prospect Heights'),
    # Aeropuertos
    '132': (40.6413, -73.7781, 'JFK Airport'),
    '138': (40.7769, -73.8740, 'LaGuardia Airport'),
    '1':   (40.6895, -74.1745, 'Newark Airport'),
    # Queens
    '7':   (40.7600, -73.8300, 'Astoria'),
    '145': (40.7440, -73.9500, 'Long Island City'),
    '82':  (40.7350, -73.8180, 'Elmhurst'),
    '93':  (40.7590, -73.8330, 'Flushing'),
    '129': (40.7480, -73.8830, 'Jackson Heights'),
    '130': (40.7020, -73.7900, 'Jamaica'),
    '70':  (40.7610, -73.8600, 'East Elmhurst'),
}

print(f"Centroides definidos para {len(ZONE_CENTROIDS)} zonas")

In [ ]:
# Agregar datos por zona para el mapa
zone_agg = df.groupby('pickup_location_id').agg(
    trip_count=('fare_amount', 'count'),
    avg_fare=('fare_amount', 'mean'),
    avg_tip_pct=('tip_percentage', 'mean'),
    avg_distance=('trip_distance', 'mean'),
    pickup_zone=('pickup_zone', 'first')
).reset_index()

# Asignar coordenadas desde el lookup
zone_agg['lat'] = zone_agg['pickup_location_id'].apply(
    lambda x: ZONE_CENTROIDS[str(x)][0] if str(x) in ZONE_CENTROIDS else np.nan
)
zone_agg['lon'] = zone_agg['pickup_location_id'].apply(
    lambda x: ZONE_CENTROIDS[str(x)][1] if str(x) in ZONE_CENTROIDS else np.nan
)
zone_agg['zone_name'] = zone_agg['pickup_location_id'].apply(
    lambda x: ZONE_CENTROIDS[str(x)][2] if str(x) in ZONE_CENTROIDS else 'Desconocida'
)

# Filtrar zonas sin centroide y con suficientes viajes
df_map = zone_agg.dropna(subset=['lat', 'lon']).copy()
df_map = df_map[df_map['trip_count'] >= 10]

# Limitar tarifa para escala de color
df_map['fare_display'] = df_map['avg_fare'].clip(upper=40)

print(f"Zonas con centroide asignado y >= 10 viajes: {len(df_map)}")
df_map.head()

In [ ]:
# Scatter Mapbox de zonas: tamaño = volumen de viajes, color = tarifa promedio
fig = px.scatter_mapbox(
    df_map,
    lat='lat',
    lon='lon',
    size='trip_count',
    size_max=30,
    color='fare_display',
    color_continuous_scale='Viridis',
    mapbox_style='open-street-map',
    zoom=10,
    center=dict(lat=40.7128, lon=-74.0060),
    opacity=0.7,
    hover_data={
        'zone_name': True,
        'pickup_zone': True,
        'trip_count': True,
        'avg_fare': ':.2f',
        'avg_tip_pct': ':.1f',
        'avg_distance': ':.1f',
        'fare_display': False,
        'lat': False,
        'lon': False
    },
    labels={
        'fare_display': 'Tarifa prom (USD)',
        'trip_count': 'Viajes',
        'avg_fare': 'Tarifa prom',
        'avg_tip_pct': 'Propina prom (%)',
        'avg_distance': 'Distancia prom (mi)',
        'zone_name': 'Zona',
        'pickup_zone': 'Borough'
    },
    title='Zonas de recogida: tamaño = volumen, color = tarifa promedio'
)

fig.update_layout(height=600, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

**Interacción:**
- Haz **zoom** con la rueda del ratón para explorar zonas específicas
- Pasa el cursor sobre un marcador para ver las estadísticas de la zona
- Los marcadores más grandes representan zonas con más viajes
- Los colores más claros (amarillos) indican tarifas promedio más altas

---
## 4. Serie temporal con rangeslider

El **rangeslider** permite seleccionar un rango de fechas arrastrando los bordes del selector inferior. Es ideal para explorar tendencias temporales.

In [ ]:
# Agregar viajes por día
daily_trips = df.groupby('pickup_date').agg(
    viajes=('fare_amount', 'count'),
    tarifa_media=('fare_amount', 'mean'),
    distancia_media=('trip_distance', 'mean')
).reset_index()

daily_trips = daily_trips.sort_values('pickup_date')

# Serie temporal con rangeslider
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_trips['pickup_date'],
    y=daily_trips['viajes'],
    mode='lines+markers',
    name='Viajes diarios',
    line=dict(color='#2196F3', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x|%Y-%m-%d}<br>Viajes: %{y}<extra></extra>'
))

# Agregar media móvil de 7 días
daily_trips['media_movil_7d'] = daily_trips['viajes'].rolling(window=7, center=True).mean()

fig.add_trace(go.Scatter(
    x=daily_trips['pickup_date'],
    y=daily_trips['media_movil_7d'],
    mode='lines',
    name='Media móvil (7 días)',
    line=dict(color='#FF5722', width=3, dash='dash'),
    hovertemplate='Fecha: %{x|%Y-%m-%d}<br>Media 7d: %{y:.0f}<extra></extra>'
))

fig.update_layout(
    title='Viajes diarios en taxi — Arrastra el slider para explorar',
    xaxis_title='Fecha',
    yaxis_title='Número de viajes (muestra)',
    template='plotly_white',
    height=500,
    xaxis=dict(
        rangeslider=dict(visible=True),
        type='date'
    ),
    legend=dict(x=0.01, y=0.99)
)

fig.show()

**Cómo usar el rangeslider:**
- Arrastra los bordes izquierdo/derecho del selector inferior para cambiar el rango de fechas
- Arrastra el centro del selector para desplazarte en el tiempo
- La media móvil suaviza las fluctuaciones y revela la tendencia subyacente

---
## 5. Dashboard con subplots: 4 gráficos en uno

Combinamos múltiples visualizaciones en una sola figura usando `make_subplots`. Esto permite crear un mini-dashboard dentro del notebook.

In [ ]:
# Dashboard con 4 subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Distribución de tarifas',
        'Viajes por hora del día',
        'Distancia vs Tarifa',
        'Propina por zona'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# 1. Histograma de tarifas
fig.add_trace(
    go.Histogram(
        x=df[df['fare_amount'] <= 50]['fare_amount'],
        nbinsx=40,
        marker_color='#2196F3',
        name='Tarifas',
        showlegend=False
    ),
    row=1, col=1
)

# 2. Viajes por hora
hourly = df.groupby('pickup_hour').size().reset_index(name='viajes')
fig.add_trace(
    go.Bar(
        x=hourly['pickup_hour'],
        y=hourly['viajes'],
        marker_color='#FF9800',
        name='Por hora',
        showlegend=False
    ),
    row=1, col=2
)

# 3. Scatter distancia vs tarifa
sample_dash = df.sample(n=3000, random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample_dash['trip_distance'],
        y=sample_dash['fare_amount'],
        mode='markers',
        marker=dict(size=3, color='#4CAF50', opacity=0.3),
        name='Dist vs Tarifa',
        showlegend=False
    ),
    row=2, col=1
)

# 4. Box plot de propina por zona
main_zones = ['Manhattan', 'Brooklyn', 'JFK', 'LaGuardia', 'Queens/Otro']
df_zones = df[(df['pickup_zone'].isin(main_zones)) &
              (df['tip_percentage'].between(0, 40))]

colors_zones = ['#E91E63', '#2196F3', '#4CAF50', '#9C27B0', '#FF9800']
for zone, color in zip(main_zones, colors_zones):
    zone_data = df_zones[df_zones['pickup_zone'] == zone]['tip_percentage']
    fig.add_trace(
        go.Box(
            y=zone_data,
            name=zone,
            marker_color=color,
            showlegend=False
        ),
        row=2, col=2
    )

# Layout
fig.update_xaxes(title_text='Tarifa (USD)', row=1, col=1)
fig.update_xaxes(title_text='Hora', row=1, col=2)
fig.update_xaxes(title_text='Distancia (mi)', row=2, col=1)
fig.update_yaxes(title_text='Frecuencia', row=1, col=1)
fig.update_yaxes(title_text='Viajes', row=1, col=2)
fig.update_yaxes(title_text='Tarifa (USD)', row=2, col=1)
fig.update_yaxes(title_text='Propina (%)', row=2, col=2)

fig.update_layout(
    title_text='Dashboard NYC Taxi — Resumen de métricas clave',
    template='plotly_white',
    height=800,
    showlegend=False
)

fig.show()

**Ventajas del dashboard con subplots:**
- Vista panorámica de múltiples métricas en un solo vistazo
- Cada subplot conserva su interactividad independiente
- Útil para presentaciones ejecutivas donde se necesita contexto rápido

---
## 6. Box plot interactivo con estadísticas por zona

Los box plots de Plotly muestran información detallada al pasar el cursor: mediana, Q1, Q3, valores atípicos.

In [ ]:
# Preparar datos con estadísticas de zona para hover
zone_stats = df[df['pickup_zone'].isin(main_zones)].groupby('pickup_zone').agg(
    tarifa_media=('fare_amount', 'mean'),
    distancia_media=('trip_distance', 'mean'),
    viajes_total=('fare_amount', 'count'),
    propina_media=('tip_percentage', 'mean')
).round(2)

df_box = df[(df['pickup_zone'].isin(main_zones)) &
            (df['fare_amount'] <= 80)].copy()

# Agregar estadísticas de zona como columnas para hover
df_box = df_box.merge(
    zone_stats, left_on='pickup_zone', right_index=True, how='left'
)

fig = px.box(
    df_box,
    x='pickup_zone',
    y='fare_amount',
    color='pickup_zone',
    color_discrete_sequence=px.colors.qualitative.Set2,
    hover_data={
        'tarifa_media': ':.2f',
        'distancia_media': ':.1f',
        'viajes_total': True,
        'propina_media': ':.1f',
        'pickup_zone': False
    },
    labels={
        'fare_amount': 'Tarifa (USD)',
        'pickup_zone': 'Zona de recogida',
        'tarifa_media': 'Tarifa media zona',
        'distancia_media': 'Distancia media (mi)',
        'viajes_total': 'Total viajes zona',
        'propina_media': 'Propina media (%)'
    },
    title='Distribución de tarifa por zona con estadísticas al hover'
)

fig.update_layout(
    template='plotly_white',
    height=500,
    showlegend=False,
    xaxis_title='Zona de recogida',
    yaxis_title='Tarifa (USD)'
)

fig.show()

**Interpretación del box plot interactivo:**
- La **caja** contiene el 50% central de los datos (Q1 a Q3)
- La **línea** dentro de la caja es la mediana
- Los **bigotes** se extienden hasta 1.5 * IQR
- Los **puntos** fuera de los bigotes son valores atípicos
- Al hacer hover, se ven las estadísticas agregadas de la zona

---
## 7. Gráfico Sunburst: jerarquía zona - tipo de pago - categoría de propina

El **sunburst** es un gráfico jerárquico circular que muestra cómo se descomponen los datos en niveles sucesivos. Es excelente para visualizar proporciones dentro de proporciones.

### Jerarquía
1. **Nivel exterior**: Zona de recogida (borough)
2. **Nivel medio**: Tipo de pago (tarjeta, efectivo, etc.)
3. **Nivel interior**: Categoría de propina

In [ ]:
# Preparar datos para sunburst
# Mapear payment_type
payment_map = {
    'CRD': 'Tarjeta',
    'CSH': 'Efectivo',
    'NOC': 'Sin cargo',
    'DIS': 'Disputa',
    'UNK': 'Desconocido',
    'Credit': 'Tarjeta',
    'Cash': 'Efectivo',
    'No Charge': 'Sin cargo',
    'Dispute': 'Disputa'
}

df_sun = df[df['pickup_zone'].isin(main_zones)].copy()
df_sun['payment_name'] = df_sun['payment_type'].map(payment_map).fillna('Otro')
df_sun = df_sun.dropna(subset=['tip_category'])

# Agregar datos
sunburst_data = df_sun.groupby(
    ['pickup_zone', 'payment_name', 'tip_category']
).size().reset_index(name='count')

# Filtrar grupos muy pequeños para legibilidad
sunburst_data = sunburst_data[sunburst_data['count'] >= 5]

fig = px.sunburst(
    sunburst_data,
    path=['pickup_zone', 'payment_name', 'tip_category'],
    values='count',
    color='count',
    color_continuous_scale='Blues',
    title='Jerarquía: Zona -> Tipo de pago -> Categoría de propina'
)

fig.update_layout(
    height=650,
    margin=dict(t=50, l=0, r=0, b=0)
)

fig.update_traces(
    hovertemplate='<b>%{label}</b><br>Viajes: %{value:,}<br>Porcentaje: %{percentParent:.1%}<extra></extra>'
)

fig.show()

**Cómo leer el sunburst:**
- **Click** en un segmento para hacer zoom a ese nivel
- **Click en el centro** para volver al nivel superior
- El **tamaño** de cada segmento es proporcional al número de viajes
- Observa que los pagos en efectivo casi nunca registran propina (el taxímetro no la captura)
- Los pagos con tarjeta muestran toda la distribución de categorías de propina

---
## Resumen del notebook

### Visualizaciones interactivas creadas

| Gráfico | Tipo | Interacción principal |
|---------|------|-----------------------|
| Histograma animado | px.histogram | Slider de hora |
| Heatmap hora x día | go.Heatmap | Hover con valores |
| Mapa de zonas | px.scatter_mapbox | Zoom, pan, hover |
| Serie temporal | go.Scatter | Rangeslider de fechas |
| Dashboard 4-en-1 | make_subplots | Zoom independiente |
| Box plot por zona | px.box | Hover con estadísticas |
| Sunburst jerárquico | px.sunburst | Click para drill-down |

### Cuándo usar cada tipo
- **Plotly Express**: Exploración rápida, prototipos, notebooks educativos
- **Graph Objects**: Dashboards de producción, personalización avanzada
- **Mapbox**: Datos geoespaciales (usar `open-street-map` para evitar tokens)
- **Sunburst/Treemap**: Datos jerárquicos con proporciones

### Enfoque basado en Taxi Zones
El mapa scatter_mapbox utiliza centroides de Taxi Zones en lugar de coordenadas GPS individuales. Cada marcador representa una zona agregada con tamaño proporcional al volumen de viajes y color según la tarifa promedio.

### Fin de la Fase 2
Con estos tres notebooks (04, 05, 06) hemos completado la fase de análisis exploratorio avanzado. En la Fase 3 comenzaremos con **ingeniería de características** y preparación para modelado.